In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# https://www.kaggle.com/competitions/tweet-sentiment-extraction/overview
# 3 emotions, approx 30k labelled tweets
df_train = pd.read_csv("/content/train.csv",encoding='ISO-8859-1',header=None)

In [3]:
df_train.drop(columns=[2,0],inplace=True)

In [4]:
df_test = pd.read_csv("/content/test.csv",encoding='ISO-8859-1',header=None)

In [5]:
df_test.drop(columns=[0],inplace=True)

In [6]:
df_test.drop([df_test.index[0]],inplace=True)
df_train.drop([df_train.index[0]],inplace=True)

In [7]:
df_test.columns = ['text','Sentiment']
df_train.columns = ['text','Sentiment']

In [8]:
df_train.head()

,text,Sentiment
1,"I`d have responded, if I were going",neutral
2,Sooo SAD I will miss you here in San Diego!!!,negative
3,my boss is bullying me...,negative
4,what interview! leave me alone,negative
5,"Sons of ****, why couldn`t they put them on t...",negative


In [9]:
df_test.head()

,text,Sentiment
1,Last session of the day http://twitpic.com/67ezh,neutral
2,Shanghai is also really exciting (precisely -...,positive
3,"Recession hit Veronique Branquinho, she has to...",negative
4,happy bday!,positive
5,http://twitpic.com/4w75p - I like it!!,positive


In [10]:
df_train.dropna(axis=0,inplace=True)
df_test.dropna(axis=0,inplace=True)

In [11]:
# Function to Clean the Tweet.

import re
def clean_tweet(tweet):
    return ' '.join(re.sub('(\\\\n)|(b\"[^0-9A-Za-z A-Za-z0-9 \t]+)|(b\'[^0-9A-Za-z]+)|(b\"[A-Za-z0-9]+)|(b\'[A-Za-z0-9]+)|(b\'#[A-Za-z0-9]+)|(b\'@[A-Za-z0-9]+)|(\\\\x[A-Za-z0-9]+)|(@[A-Za-z0-9]+)|([^0-9A-Za-z \t])|(\w+:\/\/\S+)|([RT])', ' ', str(tweet).lower()).split())


In [12]:
# Call function to get Clean tweets
df_test["CleanTweet"] = df_test['text'].apply(lambda x : clean_tweet(x))
df_train["CleanTweet"] = df_train['text'].apply(lambda x : clean_tweet(x))

In [13]:
df_test.drop(df_test.index[df_test.CleanTweet.eq("")], inplace=True)
df_train.drop(df_train.index[df_train.CleanTweet.eq("")], inplace=True)

In [14]:
#Emotion dictionary lookup
emotions_dict = {"negative":0, "neutral":2, "positive":4}
emotions_dict

def emo_lookup(emo):
  return emotions_dict[emo]

df_test['Label'] = df_test.Sentiment.apply(emo_lookup)
df_train['Label'] = df_train.Sentiment.apply(emo_lookup)

In [15]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize
from gensim.parsing.preprocessing import STOPWORDS
def preprocess(df):
	'''Function to preprocess and create corpus'''
	new_corpus=[]
	lem=WordNetLemmatizer()
	for text in df["text"]:
		words=[w for w in word_tokenize(text) if (w not in STOPWORDS)]
		words=[lem.lemmatize(w) for w in words]
		new_corpus.append(words)
	return new_corpus

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [16]:
corpus=preprocess(df_train)

In [40]:
!pip3 install -U gensim

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 24.1 MB 62.6 MB/s 
  Attempting uninstall: gensim
    Found existing installation: gensim 3.6.0
    Uninstalling gensim-3.6.0:
      Successfully uninstalled gensim-3.6.0


In [22]:
!pip3 install nlpia

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 32.0 MB 41 kB/s 
     |████████████████████████████████| 706 kB 55.1 MB/s 
     |████████████████████████████████| 120 kB 58.2 MB/s 
     |████████████████████████████████| 1.6 MB 42.7 MB/s 
     |████████████████████████████████| 210 kB 48.7 MB/s 
     |████████████████████████████████| 170 kB 45.5 MB/s 
     |████████████████████████████████| 2.2 MB 46.2 MB/s 
     |████████████████████████████████| 82 kB 739 kB/s 


In [32]:
from gensim.test.utils import datapath, get_tmpfile
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

glove_file = datapath('/content/glove.6B.100d.txt')
word2vec_glove_file = get_tmpfile("glove.6B.100d.word2vec.txt")
glove2word2vec(glove_file, word2vec_glove_file)

def load_w2v():
    word2vecDict = KeyedVectors.load_word2vec_format(word2vec_glove_file)
    embeddings_index = dict()
    for word in word2vecDict.wv.vocab:
        embeddings_index[word] = word2vecDict.word_vec(word)

    return embeddings_index


/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:7: DeprecationWarning: Call to deprecated `glove2word2vec` (KeyedVectors.load_word2vec_format(.., binary=False, no_header=True) loads GLoVE text vectors.).
  import sys


In [ ]:
#w2v_model=load_w2v()

In [34]:
def load_glove():
    embedding_dict = {}
    path = '/content/glove.6B.100d.txt'
    with open(path, 'r') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vectors = np.asarray(values[1:], 'float32')
            embedding_dict[word] = vectors
    f.close()
    return embedding_dict

In [35]:
embeddings_index = load_glove()

In [38]:
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
MAX_LEN=50
tokenizer_obj=Tokenizer()
tokenizer_obj.fit_on_texts(corpus)
sequences=tokenizer_obj.texts_to_sequences(corpus)

tweet_pad=pad_sequences(sequences,
                        maxlen=MAX_LEN,
                        truncating='post',
                        padding='post')

In [39]:
word_index=tokenizer_obj.word_index
print('Number of unique words:',len(word_index))

Number of unique words: 27034


In [60]:
from tqdm import tqdm
def prepare_matrix(embedding_dict, emb_size=100):
    num_words = len(word_index)
    embedding_matrix = np.zeros((num_words, emb_size))

    for word, i in tqdm(word_index.items()):
        #if i > num_words:
        #  continue
        emb_vec = embedding_dict.get(word)
        if emb_vec is not None:
          embedding_matrix[i] = emb_vec

    return embedding_matrix

In [45]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Embedding, GlobalAveragePooling1D

def new_model(embedding_matrix):
    inp = Input(shape=(MAX_LEN,))

    x = Embedding(num_words, embedding_matrix.shape[1], weights=[embedding_matrix],
                  trainable=False)(inp)

    x = Bidirectional(
        LSTM(60, return_sequences=True, name='lstm_layer', 
             dropout=0.1, recurrent_dropout=0.1))(x)

    x = GlobalAveragePool1D()(x)
    x = Dense(3, activation="softmax")(x)
    model = Model(inputs=inp, outputs=x)
    model.compile(loss='categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

    return model

In [42]:
df_train.head()

,text,Sentiment,CleanTweet,Label
1,"I`d have responded, if I were going",neutral,i d have responded if i were going,2
2,Sooo SAD I will miss you here in San Diego!!!,negative,sooo sad i will miss you here in san diego,0
3,my boss is bullying me...,negative,my boss is bullying me,0
4,what interview! leave me alone,negative,what interview leave me alone,0
5,"Sons of ****, why couldn`t they put them on t...",negative,sons of why couldn t they put them on the rele...,0


In [ ]:
X_train = df_train.CleanTweet
y_train = df_train.Label

In [43]:
df_test.head()

,text,Sentiment,CleanTweet,Label
1,Last session of the day http://twitpic.com/67ezh,neutral,last session of the day,2
2,Shanghai is also really exciting (precisely -...,positive,shanghai is also really exciting precisely sky...,4
3,"Recession hit Veronique Branquinho, she has to...",negative,recession hit veronique branquinho she has to ...,0
4,happy bday!,positive,happy bday,4
5,http://twitpic.com/4w75p - I like it!!,positive,i like it,4


In [44]:
X_test = df_test.CleanTweet
y_test = df_test.Label

In [61]:
embedding_matrix=prepare_matrix(embeddings_index)

100%|█████████▉| 27033/27034 [00:00<00:00, 383395.51it/s]


IndexError: ignored

In [47]:
model=new_model(embedding_matrix)

NameError: ignored

In [ ]:
history=model.fit(X_train,y_train,
                  batch_size=8,
                  epochs=5,
                  validation_data=(X_test,y_test),
                  verbose=2)